# 1. Setup and Database Connection

In [1]:
from pathlib import Path

import duckdb

In [2]:
current_dir = Path.cwd().resolve()
project_name = 'ride-hailing-demand-fleet-allocation'

if current_dir.name == project_name:
    project_root = current_dir
elif current_dir.name == 'notebooks' and current_dir.parent.name == project_name:
    project_root = current_dir.parent
else:
    project_root = current_dir / project_name

if not project_root.exists():
    raise FileNotFoundError(f'Project folder not found: {project_root}')

raw_dir = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

In [3]:
trip_files = sorted(raw_dir.glob('fhvhv_tripdata_2024-*.parquet'))
zone_file = raw_dir / 'taxi_zone_lookup.csv'
parquet_pattern = (raw_dir / 'fhvhv_tripdata_2024-*.parquet').as_posix()
database_path = processed_dir / 'nyc_taxi.db'

database_already_exists = database_path.exists()
con = duckdb.connect(database_path.as_posix())

print(f'Trip files found       : {len(trip_files)}')
print(f'Zone lookup exists     : {zone_file.exists()}')
print(f'Database path          : {database_path}')
print(f'Database already exists: {database_already_exists}')

Trip files found       : 12
Zone lookup exists     : True
Database path          : D:\My Journey\Data Project\ride-hailing-demand-fleet-allocation\data\processed\nyc_taxi.db
Database already exists: True


# 2. Create Zone Dimension

In [5]:
con.execute(f"""
    CREATE OR REPLACE TABLE dim_zone AS
    SELECT
        LocationID,
        Borough,
        Zone,
        service_zone
    FROM read_csv_auto('{zone_file.as_posix()}')
""");

In [6]:
df_dim_zone_preview = con.execute("""
    SELECT *
    FROM dim_zone
    ORDER BY LocationID
    LIMIT 10
""").df()

df_dim_zone_preview

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
6,7,Queens,Astoria,Boro Zone
7,8,Queens,Astoria Park,Boro Zone
8,9,Queens,Auburndale,Boro Zone
9,10,Queens,Baisley Park,Boro Zone


In [14]:
df_dim_zone_check = con.execute("""
    SELECT
        COUNT(*) AS total_zones,
        COUNT(DISTINCT LocationID) AS unique_zone_ids,
        COUNT(*) FILTER (
            WHERE Borough IS NULL
            OR Zone IS NULL
        ) AS missing_zone_information
    FROM dim_zone
""").df()

df_dim_zone_check

,total_zones,unique_zone_ids,missing_zone_information
0,265,265,0


# 3. Create Base Dimension

In [17]:
df_base_source = con.execute(f"""
    SELECT
        hvfhs_license_num,
        dispatching_base_num,
        COUNT(*) AS total_trips
    FROM read_parquet('{parquet_pattern}')
    GROUP BY
        hvfhs_license_num,
        dispatching_base_num
    ORDER BY hvfhs_license_num
""").df()

df_base_source

,hvfhs_license_num,dispatching_base_num,total_trips
0,HV0003,B03404,179125798
1,HV0005,B03406,60344650


In [7]:
con.execute(f"""
    CREATE OR REPLACE TABLE dim_base AS
    SELECT DISTINCT
        hvfhs_license_num,
        dispatching_base_num,
        CASE
            WHEN hvfhs_license_num = 'HV0003' THEN 'Uber'
            WHEN hvfhs_license_num = 'HV0005' THEN 'Lyft'
            ELSE 'Unknown'
        END AS platform_name
    FROM read_parquet('{parquet_pattern}')
    WHERE hvfhs_license_num IS NOT NULL
      AND dispatching_base_num IS NOT NULL
""");

In [8]:
df_dim_base_preview = con.execute("""
    SELECT *
    FROM dim_base
    ORDER BY hvfhs_license_num
""").df()

df_dim_base_preview

,hvfhs_license_num,dispatching_base_num,platform_name
0,HV0003,B03404,Uber
1,HV0005,B03406,Lyft


In [9]:
df_dim_base_check = con.execute("""
    SELECT
        COUNT(*) AS total_base_rows,
        COUNT(*) FILTER (
            WHERE hvfhs_license_num IS NULL
            OR dispatching_base_num IS NULL
        ) AS missing_base_keys,
        COUNT(*) FILTER (
            WHERE platform_name = 'Unknown'
        ) AS unknown_platforms
    FROM dim_base
""").df()

df_dim_base_check

,total_base_rows,missing_base_keys,unknown_platforms
0,2,0,0


# 4. Create Time Dimension

In [10]:
con.execute(f"""
    CREATE OR REPLACE TABLE dim_time AS
    SELECT DISTINCT
        DATE_TRUNC('hour', pickup_datetime) AS time_id,
        CAST(pickup_datetime AS DATE) AS calendar_date,
        EXTRACT(YEAR FROM pickup_datetime) AS year,
        EXTRACT(MONTH FROM pickup_datetime) AS month,
        EXTRACT(DAY FROM pickup_datetime) AS day_of_month,
        EXTRACT(HOUR FROM pickup_datetime) AS hour_of_day,
        ISODOW(pickup_datetime) AS day_of_week,
        CASE
            WHEN ISODOW(pickup_datetime) IN (6, 7) THEN 1
            ELSE 0
        END AS is_weekend
    FROM read_parquet('{parquet_pattern}')
    WHERE pickup_datetime >= TIMESTAMP '2024-01-01'
      AND pickup_datetime < TIMESTAMP '2025-01-01'
""");

In [11]:
df_dim_time_preview = con.execute("""
    SELECT *
    FROM dim_time
    ORDER BY time_id
    LIMIT 10
""").df()

df_dim_time_preview

,time_id,calendar_date,year,month,day_of_month,hour_of_day,day_of_week,is_weekend
0,2024-01-01 00:00:00,2024-01-01,2024,1,1,0,1,0
1,2024-01-01 01:00:00,2024-01-01,2024,1,1,1,1,0
2,2024-01-01 02:00:00,2024-01-01,2024,1,1,2,1,0
3,2024-01-01 03:00:00,2024-01-01,2024,1,1,3,1,0
4,2024-01-01 04:00:00,2024-01-01,2024,1,1,4,1,0
5,2024-01-01 05:00:00,2024-01-01,2024,1,1,5,1,0
6,2024-01-01 06:00:00,2024-01-01,2024,1,1,6,1,0
7,2024-01-01 07:00:00,2024-01-01,2024,1,1,7,1,0
8,2024-01-01 08:00:00,2024-01-01,2024,1,1,8,1,0
9,2024-01-01 09:00:00,2024-01-01,2024,1,1,9,1,0


In [12]:
df_dim_time_check = con.execute("""
    SELECT
        COUNT(*) AS total_time_rows,
        COUNT(DISTINCT time_id) AS unique_time_ids,
        MIN(time_id) AS min_time_id,
        MAX(time_id) AS max_time_id,
        COUNT(*) FILTER (
            WHERE is_weekend NOT IN (0, 1)
        ) AS invalid_weekend_values
    FROM dim_time
""").df()

df_dim_time_check

,total_time_rows,unique_time_ids,min_time_id,max_time_id,invalid_weekend_values
0,8784,8784,2024-01-01,2024-12-31 23:00:00,0


# 5. Create Trip Fact Table

In [13]:
con.execute(f"""
    CREATE OR REPLACE TABLE fact_trip AS
    SELECT
        hvfhs_license_num,
        dispatching_base_num,
        request_datetime,
        pickup_datetime,
        dropoff_datetime,
        DATE_TRUNC('hour', pickup_datetime) AS time_id,
        PULocationID,
        DOLocationID,
        trip_miles,
        trip_time
    FROM read_parquet('{parquet_pattern}')
    WHERE dropoff_datetime > pickup_datetime
      AND trip_miles > 0
      AND trip_time > 0
""");

In [14]:
total_fact_trip = con.execute("""
    SELECT COUNT(*)
    FROM fact_trip
""").fetchone()[0]

print(f'Total fact_trip rows: {total_fact_trip:,}')

Total fact_trip rows: 239,426,737


In [15]:
df_fact_schema = con.execute("""
    DESCRIBE fact_trip
""").df()

df_fact_schema

,column_name,column_type,null,key,default,extra
0,hvfhs_license_num,VARCHAR,YES,None,None,None
1,dispatching_base_num,VARCHAR,YES,None,None,None
2,request_datetime,TIMESTAMP,YES,None,None,None
3,pickup_datetime,TIMESTAMP,YES,None,None,None
4,dropoff_datetime,TIMESTAMP,YES,None,None,None
5,time_id,TIMESTAMP,YES,None,None,None
6,PULocationID,INTEGER,YES,None,None,None
7,DOLocationID,INTEGER,YES,None,None,None
8,trip_miles,DOUBLE,YES,None,None,None
9,trip_time,BIGINT,YES,None,None,None


In [16]:
df_fact_trip_preview = con.execute("""
    SELECT *
    FROM fact_trip
    ORDER BY time_id
    LIMIT 10
""").df()

df_fact_trip_preview

,hvfhs_license_num,dispatching_base_num,request_datetime,pickup_datetime,dropoff_datetime,time_id,PULocationID,DOLocationID,trip_miles,trip_time
0,HV0003,B03404,2024-01-01 00:21:47,2024-01-01 00:28:08,2024-01-01 01:05:39,2024-01-01,161,158,2.83,2251
1,HV0003,B03404,2024-01-01 00:10:56,2024-01-01 00:12:53,2024-01-01 00:20:05,2024-01-01,137,79,1.57,432
2,HV0003,B03404,2024-01-01 00:20:04,2024-01-01 00:23:05,2024-01-01 00:35:16,2024-01-01,79,186,1.98,731
3,HV0003,B03404,2024-01-01 00:35:46,2024-01-01 00:41:04,2024-01-01 00:56:34,2024-01-01,234,148,1.99,930
4,HV0003,B03404,2024-01-01 00:48:19,2024-01-01 00:57:21,2024-01-01 01:10:02,2024-01-01,148,97,2.65,761
5,HV0003,B03404,2024-01-01 00:03:47,2024-01-01 00:06:15,2024-01-01 00:27:53,2024-01-01,255,95,7.02,1298
6,HV0003,B03404,2024-01-01 00:22:51,2024-01-01 00:29:47,2024-01-01 00:50:08,2024-01-01,95,212,11.33,1221
7,HV0003,B03404,2024-01-01 00:45:34,2024-01-01 00:57:50,2024-01-01 01:11:27,2024-01-01,213,47,3.43,817
8,HV0003,B03404,2024-01-01 00:11:51,2024-01-01 00:16:00,2024-01-01 00:28:13,2024-01-01,209,114,1.54,733
9,HV0003,B03404,2024-01-01 00:26:48,2024-01-01 00:33:15,2024-01-01 00:46:39,2024-01-01,113,209,1.72,804


# 6. Validate Warehouse

In [17]:
# Cleaning and null check
df_fact_quality_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE dropoff_datetime <= pickup_datetime
        ) AS invalid_timestamp,
        COUNT(*) FILTER (
            WHERE trip_miles <= 0
        ) AS invalid_distance,
        COUNT(*) FILTER (
            WHERE trip_time <= 0
        ) AS invalid_duration,
        COUNT(*) FILTER (
            WHERE hvfhs_license_num IS NULL
            OR dispatching_base_num IS NULL
            OR pickup_datetime IS NULL
            OR dropoff_datetime IS NULL
            OR PULocationID IS NULL
            OR DOLocationID IS NULL
            OR trip_miles IS NULL
            OR trip_time IS NULL
        ) AS missing_critical_values
    FROM fact_trip
""").df()

df_fact_quality_check

,total_rows,invalid_timestamp,invalid_distance,invalid_duration,missing_critical_values
0,239426737,0,0,0,0


In [18]:
# Relationship check
df_relationship_check = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE pickup_zone.LocationID IS NULL
        ) AS invalid_pickup_zone,
        COUNT(*) FILTER (
            WHERE dropoff_zone.LocationID IS NULL
        ) AS invalid_dropoff_zone,
        COUNT(*) FILTER (
            WHERE base.dispatching_base_num IS NULL
        ) AS invalid_base,
        COUNT(*) FILTER (
            WHERE time.time_id IS NULL
        ) AS invalid_time
    FROM fact_trip AS trip
    LEFT JOIN dim_zone AS pickup_zone
        ON trip.PULocationID = pickup_zone.LocationID
    LEFT JOIN dim_zone AS dropoff_zone
        ON trip.DOLocationID = dropoff_zone.LocationID
    LEFT JOIN dim_base AS base
        ON trip.hvfhs_license_num = base.hvfhs_license_num
        AND trip.dispatching_base_num = base.dispatching_base_num
    LEFT JOIN dim_time AS time
        ON trip.time_id = time.time_id
""").df()

df_relationship_check

,invalid_pickup_zone,invalid_dropoff_zone,invalid_base,invalid_time
0,0,0,0,0


In [19]:
df_warehouse_summary = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM dim_zone) AS dim_zone_rows,
        (SELECT COUNT(*) FROM dim_base) AS dim_base_rows,
        (SELECT COUNT(*) FROM dim_time) AS dim_time_rows,
        (SELECT COUNT(*) FROM fact_trip) AS fact_trip_rows
""").df()

df_warehouse_summary

,dim_zone_rows,dim_base_rows,dim_time_rows,fact_trip_rows
0,265,2,8784,239426737


In [20]:
con.close()